In [15]:
import pandas as pd
import numpy as np
import openai
import json
import tiktoken

from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, Filter, FieldCondition, Prefetch, FusionQuery, Distance, PayloadSchemaType, VectorParams, MatchAny

In [2]:
## retrieve all item ids from amazon items Qdrant Collection (hybrid)

qdrant_client = QdrantClient(url="http://localhost:6333")

dummy_vector = np.zeros(1536).tolist()

In [7]:
payload = qdrant_client.query_points(
    collection_name = "amazon_items-collection-hybrid-02",
    query = dummy_vector,
    using = "text-embedding-3-small",
    limit = 1000,
    with_payload = ["parent_asin"],
    with_vectors = False,
)

In [8]:
payload.points

[ScoredPoint(id=248, version=3, score=0.0, payload={'parent_asin': 'B0B1MZT835'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=427, version=3, score=0.0, payload={'parent_asin': 'B0B6RC686L'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=307, version=3, score=0.0, payload={'parent_asin': 'B09V7MDRZ1'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=431, version=3, score=0.0, payload={'parent_asin': 'B0C3QYJZN5'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=58, version=3, score=0.0, payload={'parent_asin': 'B08M65W3PT'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=499, version=3, score=0.0, payload={'parent_asin': 'B0BXCF92XH'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=254, version=3, score=0.0, payload={'parent_asin': 'B09QS7W8G5'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=49, version=3, score=0.0, payload={'parent_asin': 'B09VT5WLMM'}, vector=Non

In [9]:
parent_asin_list = [item.payload["parent_asin"] for item in payload.points]
parent_asin_list

['B0B1MZT835',
 'B0B6RC686L',
 'B09V7MDRZ1',
 'B0C3QYJZN5',
 'B08M65W3PT',
 'B0BXCF92XH',
 'B09QS7W8G5',
 'B09VT5WLMM',
 'B0BGDTDV87',
 'B0C3LNX75K',
 'B08BR9X387',
 'B0C3VLL5CG',
 'B0C27X5VJD',
 'B0B8P44G8M',
 'B0C78B1BTB',
 'B0BJV3VJC6',
 'B0B5HMT1YW',
 'B09Q9C6MG1',
 'B0BYKDMMVJ',
 'B0C7Q3X76Q',
 'B0C13NF5JC',
 'B0BFPZGYLD',
 'B0C4P5X7XB',
 'B07YZHJRCV',
 'B0BN7N7GQ2',
 'B0B6894XGM',
 'B09NBS9DS6',
 'B0BGR94M7C',
 'B0C2WWCBSG',
 'B0C72KCMHN',
 'B0C5X8S42F',
 'B0B45VPMT4',
 'B09MPNX216',
 'B0B1D4WJWT',
 'B09QJRYJW4',
 'B0C4Y1JRRB',
 'B0B5H7T7XZ',
 'B0BCKCJQPN',
 'B0B1GXQ78F',
 'B0BZQ5YKKY',
 'B0BMXG42VN',
 'B0BF4STLSR',
 'B0C6LCJ95V',
 'B09PYFMTBF',
 'B0C5CLD1HF',
 'B0C6649XPG',
 'B09CCL371W',
 'B09GNFDH9M',
 'B09VNTZBPG',
 'B0BQCG3T3B',
 'B0CCC6PRK8',
 'B09W5MF81W',
 'B0B9RGGXC2',
 'B0BTDJ6FBR',
 'B0BQZ23TJL',
 'B0CF1T9H2H',
 'B09W5KD1SD',
 'B0B8CKD3TN',
 'B0C7K8XHQF',
 'B0BDZY67PK',
 'B09R4Y2HKY',
 'B09SFN9NRX',
 'B09LH466KZ',
 'B0C94HPBL2',
 'B0BLCMRKR4',
 'B0BTY2CB5X',
 'B09NQ16K

In [12]:
df_reviews = pd.read_json("../../data/Electronics_2022_onwards_with_ratings_100_sample_1000.jsonl", lines=True)
df_reviews.head()


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5,Perfect!,This is perfect! Thank you so much!!! I absolu...,[],B09992M2LX,B09ZPV8WBV,AHX4XWVVQUKT3FCNWCVASDF4Q56Q,2022-08-05 04:06:39.589,0,True
1,5,3ft mini usb cables,I don't have many things that still use a mini...,[],B09Y94B2NM,B09Y95BMKX,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2022-07-16 16:03:28.714,3,True
2,5,I would buy it again.,Great product. Worked well for what we needed ...,[],B07T55DL33,B0B2JWCMCY,AF5KFHNT3TQJ2GNSE3FCDFQOBICA,2019-12-09 22:35:00.531,0,True
3,5,Great to Have Around,My husband and I were recently working a booth...,[],B09M89JN7B,B0BYYGZHG5,AHV6QCNBJNSGLATP56JAWJ3C4G2A,2022-03-22 01:43:49.342,0,False
4,5,Easy to use,Work as advertised and at a very good price.,[],B07T55DL33,B0B2JWCMCY,AG7WKTZINOFIXMZJYIPKIB7PV7NQ,2019-12-28 06:12:24.960,0,True


In [13]:
len(df_reviews)

105918

In [15]:
df_reviews_sample = df_reviews[df_reviews["parent_asin"].isin(parent_asin_list)]
df_reviews_sample.head()
df_reviews_sample.shape

(105918, 10)

### Functions to transform reviews data

In [16]:
def transform_reviews_data(row):
    """
    Transform the reviews data into a list of dictionaries
    """
    return f"{row["title"]} {row["text"]}"

In [18]:
encoding = tiktoken.encoding_for_model("text-embedding-3-small")
encoding.encode("I am superman")

[40, 1097, 2307, 1543]

In [19]:
def token_count(row, model="text-embedding-3-small"):
    """
    Count the number of tokens in a string
    """
    encoding = tiktoken.encoding_for_model(model)
    return len(encoding.encode(row["transformed_text"]))


In [20]:
df_reviews_sample["transformed_text"] = df_reviews_sample.apply(transform_reviews_data, axis=1)

In [21]:
df_reviews_sample["transformed_text_token_count"] = df_reviews_sample.apply(token_count, axis=1)

In [22]:
df_reviews_sample.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,transformed_text,transformed_text_token_count
0,5,Perfect!,This is perfect! Thank you so much!!! I absolu...,[],B09992M2LX,B09ZPV8WBV,AHX4XWVVQUKT3FCNWCVASDF4Q56Q,2022-08-05 04:06:39.589,0,True,Perfect! This is perfect! Thank you so much!!!...,21
1,5,3ft mini usb cables,I don't have many things that still use a mini...,[],B09Y94B2NM,B09Y95BMKX,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2022-07-16 16:03:28.714,3,True,3ft mini usb cables I don't have many things t...,114
2,5,I would buy it again.,Great product. Worked well for what we needed ...,[],B07T55DL33,B0B2JWCMCY,AF5KFHNT3TQJ2GNSE3FCDFQOBICA,2019-12-09 22:35:00.531,0,True,I would buy it again. Great product. Worked we...,24
3,5,Great to Have Around,My husband and I were recently working a booth...,[],B09M89JN7B,B0BYYGZHG5,AHV6QCNBJNSGLATP56JAWJ3C4G2A,2022-03-22 01:43:49.342,0,False,Great to Have Around My husband and I were rec...,76
4,5,Easy to use,Work as advertised and at a very good price.,[],B07T55DL33,B0B2JWCMCY,AG7WKTZINOFIXMZJYIPKIB7PV7NQ,2019-12-28 06:12:24.960,0,True,Easy to use Work as advertised and at a very g...,13


In [25]:
df_reviews_sample = df_reviews_sample[df_reviews_sample["transformed_text_token_count"] < 8192]

In [26]:
len(df_reviews_sample)

105918

In [27]:
total_tokens = df_reviews_sample["transformed_text_token_count"].sum()
total_tokens

np.int64(6204437)

Create a new Qdrant Collection for reviews

In [ ]:
qdrant_client.create_collection(
    collection_name = "amazon-item-collection-hybrid-01-reviews",
    vectors_config = VectorParams(size=1536, distance=Distance.COSINE),
)

True

In [30]:
qdrant_client.create_payload_index(
    collection_name = "amazon-item-collection-hybrid-01-reviews",
    field_name = "parent_asin",
    field_schema = PayloadSchemaType.KEYWORD,
)


UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [17]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=[text],
        model=model,
    )
    return response.data[0].embedding

def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

In [32]:
data_to_embed_reviews = df_reviews_sample[["transformed_text", "parent_asin"]].to_dict(orient="records")

In [33]:
data_to_embed_reviews

[{'transformed_text': 'Perfect! This is perfect! Thank you so much!!! I absolutely love it!!! It’s great quality!!!',
  'parent_asin': 'B09ZPV8WBV'},
 {'transformed_text': "3ft mini usb cables I don't have many things that still use a mini USB charger cable, but I after a while you have to replace the charger cables.  These seem to work well and it was noted in the ad that they are reinforced at the head area to prolong life.  We shall see - time will tell.  I never hesitate to update my reviews should new info seem useful.  I do not accept any discounts or deals that are not available to all shoppers. And my reviews are based purely on my personal experience with each item I review.",
  'parent_asin': 'B09Y95BMKX'},
 {'transformed_text': 'I would buy it again. Great product. Worked well for what we needed it for. Would buy it again.',
  'parent_asin': 'B0B2JWCMCY'},
 {'transformed_text': 'Great to Have Around My husband and I were recently working a booth at a trade show for his busin

In [34]:
text_to_embed_reviews = [data["transformed_text"] for data in data_to_embed_reviews]

In [35]:
embeddings_reviews = get_embeddings_batch(text_to_embed_reviews, batch_size=500)
len(embeddings_reviews)

Processed 500 of 105918
Processed 1000 of 105918
Processed 1500 of 105918
Processed 2000 of 105918
Processed 2500 of 105918
Processed 3000 of 105918
Processed 3500 of 105918
Processed 4000 of 105918
Processed 4500 of 105918
Processed 5000 of 105918
Processed 5500 of 105918
Processed 6000 of 105918
Processed 6500 of 105918
Processed 7000 of 105918
Processed 7500 of 105918
Processed 8000 of 105918
Processed 8500 of 105918
Processed 9000 of 105918
Processed 9500 of 105918
Processed 10000 of 105918
Processed 10500 of 105918
Processed 11000 of 105918
Processed 11500 of 105918
Processed 12000 of 105918
Processed 12500 of 105918
Processed 13000 of 105918
Processed 13500 of 105918
Processed 14000 of 105918
Processed 14500 of 105918
Processed 15000 of 105918
Processed 15500 of 105918
Processed 16000 of 105918
Processed 16500 of 105918
Processed 17000 of 105918
Processed 17500 of 105918
Processed 18000 of 105918
Processed 18500 of 105918
Processed 19000 of 105918
Processed 19500 of 105918
Proces

105918

In [37]:
pointstructs = []
i = 1
for embedding, data in zip(embeddings_reviews, data_to_embed_reviews):
    pointstructs.append(
        PointStruct(
            id=i,
            vector=embedding,
            payload={
                "text": data["transformed_text"],
                "parent_asin": data["parent_asin"],
            }
        )
    )
    i += 1

In [38]:
batch_size_qdrant = 100
counter = 1
for i in range(0, len(pointstructs), batch_size_qdrant):
    batch = pointstructs[i:i + batch_size_qdrant]
    qdrant_client.upsert(
        collection_name="amazon-item-collection-hybrid-01-reviews",
        wait=True,
        points=batch
    )
    print(f"Processed {counter * batch_size_qdrant} of {len(pointstructs)}")
    counter += 1

Processed 100 of 105918
Processed 200 of 105918
Processed 300 of 105918
Processed 400 of 105918
Processed 500 of 105918
Processed 600 of 105918
Processed 700 of 105918
Processed 800 of 105918
Processed 900 of 105918
Processed 1000 of 105918
Processed 1100 of 105918
Processed 1200 of 105918
Processed 1300 of 105918
Processed 1400 of 105918
Processed 1500 of 105918
Processed 1600 of 105918
Processed 1700 of 105918
Processed 1800 of 105918
Processed 1900 of 105918
Processed 2000 of 105918
Processed 2100 of 105918
Processed 2200 of 105918
Processed 2300 of 105918
Processed 2400 of 105918
Processed 2500 of 105918
Processed 2600 of 105918
Processed 2700 of 105918
Processed 2800 of 105918
Processed 2900 of 105918
Processed 3000 of 105918
Processed 3100 of 105918
Processed 3200 of 105918
Processed 3300 of 105918
Processed 3400 of 105918
Processed 3500 of 105918
Processed 3600 of 105918
Processed 3700 of 105918
Processed 3800 of 105918
Processed 3900 of 105918
Processed 4000 of 105918
Processed

### Retrieval function of user reviews against list of product IDs

In [41]:
def retrieve_prefiltered_reviews_data(query, parent_asins, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="amazon-item-collection-hybrid-01-reviews",
        prefetch=[
            Prefetch(
                query=query_embedding,
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin",
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    return results

In [42]:
points = retrieve_prefiltered_reviews_data("cheaper", parent_asins=["B08WQ55H4Y"], k=5)
points

QueryResponse(points=[ScoredPoint(id=5772, version=60, score=0.5, payload={'text': 'cheap enough Good price', 'parent_asin': 'B08WQ55H4Y'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=10, version=3, score=0.33333334, payload={'text': 'Cheaper than an electrician! Soooooooo much cheaper than an electrician who told me to continue to use it rather than install more outlets!', 'parent_asin': 'B08WQ55H4Y'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=17582, version=178, score=0.25, payload={'text': 'good value for money Fast delivery and inexpensive.<br />I had to have that long cord.', 'parent_asin': 'B08WQ55H4Y'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=60806, version=611, score=0.2, payload={'text': 'Great Price! The best price I could find', 'parent_asin': 'B08WQ55H4Y'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=4932, version=52, score=0.16666667, payload={'text': 'Perfect It works fine, perfect for the c

In [19]:

def retrieve_reviews(query, product_ids, k=5):
    # INSERT_YOUR_CODE
    """
    Retrieve reviews for a given query and list of product IDs.

    Args:
        query (str): The query string to search relevant reviews.
        product_ids (List[str]): A list of product IDs (parent_asin) for which reviews are to be retrieved.
        k (int, optional): The number of top reviews to retrieve. Defaults to 5.

    Returns:
        dict: A dictionary containing:
            - 'retrieved_context_ids': List of product IDs corresponding to each retrieved review.
            - 'retrieved_context': List of review texts retrieved for the query and product IDs.
            - 'similarity_scores': List of similarity scores for each retrieved review.
    """
    qdrant_client = QdrantClient(url="http://localhost:6333")
    
    collection_name = "amazon-item-collection-hybrid-01-reviews"
    k=5
    
    querry_embeddings = get_embedding(query)
    
    response = qdrant_client.query_points(
        collection_name=collection_name,
        prefetch=[Prefetch(
            query=querry_embeddings,
            filter=Filter(
                must=[
                    FieldCondition(key="parent_asin", 
                                    match=MatchAny(any=product_ids))
                    
                    ]
                    
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )
    
    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []

    for result in response.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["text"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
    }

def process_reviews_context(context):

    formatted_context = ""

    for id, chunk in zip(context["retrieved_context_ids"], context["retrieved_context"]):
        formatted_context += f"- ID: {id}, review: {chunk}\n"

    return formatted_context


def get_formatted_reviews_context(query: str, item_list: list, top_k: int = 15) -> str:
    """Get the top k reviews matching a query for a list of prefiltered items.
    
    Args:
        query: The query to get the top k reviews for
        item_list: The list of item IDs to prefilter for before running the query
        top_k: The number of reviews to retrieve, this should be at least 20 if multipple items are prefiltered
    
    Returns:
        A string of the top k context chunks with IDs prepending each chunk, each representing a review for a given inventory item for a given query.
    """

    context = retrieve_reviews(query, item_list, top_k)
    formatted_context = process_reviews_context(context)

    return formatted_context

In [20]:
points = get_formatted_reviews_context("cheaper", item_list=["B08WQ55H4Y"], top_k=5)
points

'- ID: B08WQ55H4Y, review: cheap enough Good price\n- ID: B08WQ55H4Y, review: Cheaper than an electrician! Soooooooo much cheaper than an electrician who told me to continue to use it rather than install more outlets!\n- ID: B08WQ55H4Y, review: good value for money Fast delivery and inexpensive.<br />I had to have that long cord.\n- ID: B08WQ55H4Y, review: Great Price! The best price I could find\n- ID: B08WQ55H4Y, review: Perfect It works fine, perfect for the college student and super cheap.\n'